# IE643 — Cleaning: reports, labels and 512 px images

Input: **IE643 Project, Version 3** (`jithendraampatt/ie643-project`): 10,000 frontal chest X-rays with their
CheXpert Plus reports and CheXbert labels. Optionally also the output of the **IE643 EDA** notebook (`splits.csv`);
without it the same patient-level split (seed 287) is recomputed here and checked against the EDA counts.

The six steps, each motivated by a finding in the EDA:

| # | Step | Why (EDA finding) |
|---|------|-------------------|
| 1 | Clean the impression text: drop attestation / consult-line / summary-code / sign-off boilerplate and "1." list markers, lowercase, normalise spacing | 6 % of impressions carry "I have personally reviewed…", 89 % are numbered lists, 86 % upper case — none of it is visible in the image |
| 2 | Flag references to earlier studies and build a `impression_noprior` variant with them removed | "interval", "unchanged", "redemonstration"… describe images the model never sees and teach it to invent comparisons |
| 3 | Drop rows with no usable impression | 3 images have no impression |
| 4 | Count DistilGPT2 tokens; generation cap = 160 | at 128 tokens 21 % of raw impressions were truncated, at 160 only 9.6 % |
| 5 | Resize every image once to 512 × 512 (aspect kept, zero-padded, area interpolation) into one `uint8` array | the 34 GB of ~2,700 × 2,300 PNGs would make decoding the training bottleneck; 512 is the teacher's own input size and leaves room for 224/256/384 students |
| 6 | Training targets from the impression labels: positive = 1, blank/negative = 0, uncertain (−1) masked out | uncertain share is 80 % for Pneumonia and ~63 % for Consolidation / Enlarged Cardiomediastinum; Pneumonia and Pleural Other are too rare for per-class test metrics |

**Outputs** (`/kaggle/working`, attach this notebook's output as an input in later notebooks):
- `images_512.npy` — `uint8` array `(N, 512, 512)`, row *i* ↔ `idx == i` in the CSV (open with `np.load(..., mmap_mode="r")`)
- `reports_clean.csv` — one row per kept image: ids, split, raw / clean / no-prior impression, flags, token counts,
  raw CheXbert labels, `y_*` targets and `m_*` masks, resize geometry
- `dropped_rows.csv`, `cleaning_summary.json`

The teacher (CheXagent-2, 512 × 512 input) reads the **original** PNGs during caching and resizes them itself;
`images_512.npy` is the student's input.

Settings: CPU is enough (no GPU). Internet on is only used to download the DistilGPT2 tokenizer.

In [ ]:
import os, re, json, glob, time, random, unicodedata, collections
from multiprocessing import Pool
import numpy as np, pandas as pd
import cv2
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 200)
INPUT = os.environ.get("IE643_INPUT", "/kaggle/input")
WORK = os.environ.get("IE643_WORK", "/kaggle/working")
SIZE = 512          # stored resolution (student resizes down from this in the dataloader)
MAX_TOK = 160       # generation length cap chosen in the EDA
SEED, SPLIT_SEED = 42, 287
random.seed(SEED)

def find_file(name, base=INPUT, max_depth=5):
    for dirpath, dirnames, filenames in os.walk(base):
        if name in filenames:
            return os.path.join(dirpath, name)
        depth = dirpath[len(base):].count(os.sep)
        dirnames[:] = [] if depth >= max_depth else [d for d in dirnames if not d.startswith(("part_", "patient"))]
    return None

MAN = find_file("stratified_manifest.csv")
assert MAN, "Attach 'IE643 Project' (Version 3) as an input"
ROOT = os.path.dirname(MAN)
LABELS = ["No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion",
          "Edema", "Consolidation", "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion",
          "Pleural Other", "Fracture", "Support Devices"]
PATHS = [l for l in LABELS if l != "No Finding"]

df = pd.read_csv(MAN)
lab = pd.read_csv(os.path.join(ROOT, "chexbert_impression.csv"))
df = df.merge(lab[["path_to_image"] + LABELS], on="path_to_image", how="left", validate="one_to_one")
df["label"] = np.where(df["is_no_finding"].astype(bool), "Normal", "Abnormal")
df["abs_path"] = df["local_path"].map(lambda p: os.path.join(ROOT, p))
assert len(df) == 10000 and df["path_to_image"].is_unique
print("ROOT:", ROOT, "| rows:", len(df))

### Split (from the EDA)

Use `splits.csv` from the IE643 EDA output if it is attached; otherwise rebuild the identical split (patient-level,
80/10/10 by image count, stratified on any-abnormal, seed 287) and check it against the EDA's counts.

In [ ]:
def make_split(seed):
    pat = df.groupby("deid_patient_id").agg(n=("local_path", "size"),
                                             any_abn=("label", lambda s: (s == "Abnormal").any()))
    assign = {}
    for flag, g in pat.groupby("any_abn"):
        ids = g.sample(frac=1, random_state=seed).index.to_numpy()
        cum = g.loc[ids, "n"].cumsum() / g["n"].sum()
        assign.update({i: ("train" if c <= .8 else "val" if c <= .9 else "test") for i, c in zip(ids, cum)})
    return df["deid_patient_id"].map(assign)

SPL = find_file("splits.csv")
if SPL:
    s = pd.read_csv(SPL, usecols=["path_to_image", "subset"])
    df = df.merge(s, on="path_to_image", how="left", validate="one_to_one")
    print("Split taken from", SPL)
else:
    df["subset"] = make_split(SPLIT_SEED)
    print(f"No splits.csv attached — recomputed with seed {SPLIT_SEED}")
counts = df["subset"].value_counts().reindex(["train", "val", "test"])
print(counts.to_dict())
assert counts.tolist() == [7999, 1000, 1001], "split differs from the EDA (7999/1000/1001)"
assert df.groupby("deid_patient_id")["subset"].nunique().max() == 1, "patient leakage across splits"

## Step 1 — Clean the impression text

Rules are applied in this order and every rule's hit count is reported, so nothing is removed silently.
Boilerplate = text that no image could predict: radiologist attestations, the consult phone line, summary codes,
sign-offs, and "results were discussed with Dr. …" communication sentences.

In [ ]:
BOILER = [  # (name, pattern) — case-insensitive; removed wherever they occur
    ("attestation",  r"i\s+have\s+(?:personally\s+)?reviewed\s+the\s+images?\b[^.]*\.?"),
    ("consult_line", r"physician\s+to\s+physician\s+radiology\s+consult\s+line[^.]*?(?:\(?\d{3}\)?[\s.-]*\d{3}[\s.-]*\d{4})?\s*\.?"),
    ("summary_code", r"\bsummary(?:\s+codes?)?\s*:?\s*(?:\d+\s*-\s*)?(?:s\s+)?(?:no\s+significant\s+abnormality|normal|abnormal|significant\s+abnormality|minor\s+abnormality|critical|possible\s+malignancy)[^.]*\.?"),
    ("end_marker",   r"\bend\s+of\s+impression\s*:?"),
    ("signed",       r"(?:^|(?<=[.\s]))(?:electronically\s+)?signed(?:\s+by)?\s*:.*$"),
    ("phone",        r"\(?\b\d{3}\)?[\s.-]\d{3}[\s.-]\d{4}\b"),
    ("date",         r"\b(?:on\s+)?\d{1,2}[-/]\d{1,2}[-/]\d{2,4}\b"),
    ("clock_time",   r"\b(?:at\s+)?(?:\d{1,2}:\d{2}\s*(?:a\.?m\.?|p\.?m\.?)?|\d{3,4}\s*(?:hours|hrs)\b)"),
]
BOILER = [(n, re.compile(p, re.I | re.S)) for n, p in BOILER]
COMM = re.compile(r"\b(?:discussed|communicated|notified|called|paged|conveyed|relayed|reported)\b.*?"
                  r"\b(?:dr|doctor|physician|nurse|np|pa|rn|md|team|provider|clinician)\b", re.I)
LIST_MARK = re.compile(r"(?:(?<=^)|(?<=\s)|(?<=[.:]))\(?\d{1,2}[.)](?=\s*[a-z])", re.I)   # "1." / "2)" before a word, not "2.5 cm"
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")
hits = collections.Counter()

def sentences(t):
    return [s.strip() for s in SENT_SPLIT.split(t) if s.strip()]

def finish(sents):
    sents = [s.strip().rstrip(",;: ").strip() for s in sents]
    sents = [s for s in sents if s]
    t = " ".join(s if s.endswith((".", "!", "?")) else s + "." for s in sents)
    t = re.sub(r"\s+([.,;:])", r"\1", t)
    t = re.sub(r"[,;:]+\.", ".", t)
    t = re.sub(r"([.,;:])\1+", r"\1", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def clean_impression(raw):
    if not isinstance(raw, str) or not raw.strip():
        return ""
    t = unicodedata.normalize("NFKC", raw).replace("\r", " ").replace("\n", " ").replace('"', " ")
    for name, rx in BOILER:
        t, n = rx.subn(" ", t)
        hits[name] += n
    t, n = LIST_MARK.subn(" ", t); hits["list_marker"] += n
    t = re.sub(r"\b(dr|mr|mrs|ms|drs)\.\s", r"\1 ", t, flags=re.I)   # keep "Dr. X" inside one sentence
    t = re.sub(r"\s+", " ", t).strip().lower()
    keep = []
    for s in sentences(t):
        if COMM.search(s):
            hits["communication"] += 1
            continue
        if re.match(r"^[\W\d]*(?:electronically\s+)?signed\b", s):   # sign-off left as its own sentence
            hits["signed"] += 1
            continue
        if re.search(r"\bsee\s+(?:the\s+)?(?:body\s+of\s+(?:the\s+)?report|findings\s+(?:section|above)|report\s+above|above)\b", s):
            hits["see_report"] += 1                 # pointer to another section, not a finding
            continue
        if re.fullmatch(r"comparisons?\b\W*(?:none|no\s+prior.*|not\s+available)?\W*", s):   # empty comparison header
            continue
        if re.fullmatch(r"[\W\d_]*", s):          # leftover punctuation / numbers only
            continue
        keep.append(s)
    return finish(keep)

df["impression_raw"] = df["section_impression"].fillna("")
df["impression_clean"] = df["impression_raw"].map(clean_impression)
for k, v in hits.most_common():
    print(f"  {k:<14} {v:>6,} hits")
changed = (df["impression_clean"] != df["impression_raw"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip())
print(f"\nImpressions changed beyond lowercasing: {changed.mean()*100:.1f}%")

**Check what survived.** Sentences that repeat across many different reports are templates; the list below is the
place to spot boilerplate the rules missed (normal-study templates such as "no acute cardiopulmonary disease"
are legitimate findings and stay).

In [ ]:
sent_counts = collections.Counter(s for t in df["impression_clean"] for s in sentences(t))
print("Most frequent sentences after cleaning:")
for s, n in sent_counts.most_common(30):
    print(f"{n:>5}  {s[:150]}")

print("\nRandom before → after (changed reports only):")
for _, r in df[changed].sample(12, random_state=SEED).iterrows():
    print("RAW  :", r.impression_raw[:400].replace("\n", " "))
    print("CLEAN:", r.impression_clean[:400], "\n")

## Step 2 — References to earlier studies

A single image cannot show change over time. Two outputs:
- `has_prior` — the clean impression mentions a comparison (kept for analysis and for an ablation).
- `impression_noprior` — sentences that only state a comparison are dropped ("no interval change", "unchanged",
  "comparison: …"); in the rest, temporal modifiers are removed while the finding is kept
  ("stable small left effusion" → "small left effusion", "interval placement of a picc" → "placement of a picc").
  Comparative verbs that cannot be removed without rewriting the sentence ("improved", "decreased"…) are
  flagged as `noprior_residual` so training can exclude those reports if the ablation says it matters.

In [ ]:
PRIOR = re.compile(r"\b(?:prior|previous(?:ly)?|interval|unchanged|stable|again|redemonstrat\w*|persist\w*|"
                   r"compar\w*|since|today'?s|no\s+(?:significant\s+)?change|new|newly|improv\w*|worsen\w*|"
                   r"increas\w*|decreas\w*|resolv\w*|resolution|progress\w*|re-?demonstrat\w*|remain\w*)\b", re.I)
# sentences that carry nothing but the comparison
ONLY_COMP = re.compile(r"^(?:(?:comparison\b.*|(?:there\s+(?:is|has\s+been)\s+)?no\s+(?:significant\s+|substantial\s+|"
                       r"appreciable\s+)?(?:interval\s+)?change\b[^.]*|(?:otherwise\s+)?(?:unchanged|stable)"
                       r"(?:\s+(?:exam|examination|study|appearance|chest|radiograph|findings?))?\.?|"
                       r"(?:(?:the\s+)?findings\s+(?:are\s+)?)?(?:otherwise\s+)?(?:unchanged|stable)\s+(?:from|compared|since)\b.*))\.?$", re.I)
CLAUSES = [  # removed inside a sentence (comparison clause / temporal modifier), finding kept
    r"\s+(?:is|are|remains?|appears?|has\s+been|have\s+been)\s+(?:grossly\s+|essentially\s+|again\s+|overall\s+)?(?:unchanged|stable)\b",
    r",?\s*(?:as\s+)?compared\s+(?:to|with)\s+(?:the\s+)?(?:prior|previous|recent|most\s+recent)[^,.;]*",
    r",?\s*(?:when\s+)?compared\s+(?:to|with)\s+[^,.;]*?(?:study|exam|examination|radiograph|film|ct|chest)\b",
    r",?\s*(?:since|from|than\s+(?:on\s+)?)\s*(?:the\s+)?(?:prior|previous|recent|most\s+recent|last)\s+"
    r"(?:study|exam|examination|radiographs?|films?|ct|chest|day|one)?[^,.;]*",
    r",?\s*(?:as\s+)?(?:on|in)\s+(?:the\s+)?(?:prior|previous)\s+(?:study|exam|examination|radiographs?|films?)",
    r"\s+(?:or|and)\s+(?:interval\s+)?change\b",
    r"\s*(?:on|in)\s+today'?s\s+(?:study|exam|examination|radiographs?|films?)",
    r",?\s*as\s+(?:before|previously(?:\s+(?:seen|noted|described))?)",
    r"\b(?:again\s+(?:seen|noted|demonstrated|identified)|redemonstrat(?:ion|ed)\s+of|re-?demonstrated)\b",
    r"\b(?:interval(?:ly)?|unchanged|stable|persistent(?:ly)?|grossly\s+stable|previously\s+(?:seen|noted|described))\b",
]
CLAUSES = [re.compile(p, re.I) for p in CLAUSES]
RESIDUAL = re.compile(r"\b(?:prior|previous|improv\w*|worsen\w*|increas\w*|decreas\w*|resolv\w*|resolution|"
                      r"progress\w*|new|newly|no\s+longer|remain\w*|since)\b", re.I)

def remove_prior(t):
    out = []
    for s in sentences(t):
        if ONLY_COMP.match(s.strip()):
            continue
        orig = s
        for rx in CLAUSES:
            s = rx.sub(" ", s)
        s = re.sub(r"\s+", " ", s).strip(" ,;.")
        s = re.sub(r"^(?:is|are|there\s+is)\s+", "", s)
        if s and (s == orig or re.search(r"[a-z]{4,}", s)):   # drop only stubs left by removal ("of the")
            out.append(s)
    return finish(out)

df["has_prior"] = df["impression_clean"].str.contains(PRIOR)
df["impression_noprior"] = df["impression_clean"].map(remove_prior)
df["noprior_residual"] = df["impression_noprior"].str.contains(RESIDUAL)
print(f"Impressions with a comparison reference : {df.has_prior.mean()*100:.1f}%")
print(f"  … still comparative after removal     : {df.noprior_residual.mean()*100:.1f}%  (flag noprior_residual)")
print(f"  … emptied entirely by removal         : {((df.impression_noprior == '') & (df.impression_clean != '')).sum()}")
print("\nExamples (clean → noprior):")
for _, r in df[df.has_prior].sample(12, random_state=SEED).iterrows():
    print("CLEAN  :", r.impression_clean[:300])
    print("NOPRIOR:", r.impression_noprior[:300], "\n")

## Step 3 — Drop rows without a usable impression

A row is dropped if the cleaned impression is empty (the 3 missing impressions, plus any report that was pure
boilerplate). Rows whose *no-prior* variant is empty are kept but flagged — they are only unusable for the
no-prior ablation.

In [ ]:
df["noprior_empty"] = df["impression_noprior"].eq("")
drop = df["impression_clean"].eq("")
dropped = df.loc[drop, ["local_path", "path_to_image", "deid_patient_id", "subset", "impression_raw"]].assign(
    reason=np.where(df.loc[drop, "impression_raw"].str.strip().eq(""), "no impression", "only boilerplate"))
print(f"Dropped {drop.sum()} rows:"); print(dropped[["local_path", "subset", "reason"]].head(25).to_string(index=False))
df = df.loc[~drop].reset_index(drop=True)
df["idx"] = np.arange(len(df))
print("\nKept per split:", df["subset"].value_counts().reindex(["train", "val", "test"]).to_dict())
print("No-prior variant empty (kept, flagged):", int(df.noprior_empty.sum()))

## Step 4 — Token lengths (DistilGPT2) and the 160-token cap

The cap itself is applied by the training dataloader; here we record lengths so the cut is measurable and the
truncated share can be reported.

In [ ]:
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("distilgpt2")
    ntok = lambda texts: [len(x) for x in tok(list(texts), add_special_tokens=False)["input_ids"]]
    TOKENIZER = "distilgpt2"
except Exception as e:                                   # offline fallback: ~1.3 BPE tokens per word
    print("Tokenizer unavailable (", type(e).__name__, ") — using a word-count estimate")
    ntok = lambda texts: [int(round(len(t.split()) * 1.3)) for t in texts]
    TOKENIZER = "estimate (1.3 x words)"

for col in ["impression_raw", "impression_clean", "impression_noprior"]:
    df["ntok_" + col.split("_")[1]] = ntok(df[col])
q = [.5, .9, .95, .99, 1.0]
tab = pd.DataFrame({c: df["ntok_" + c].quantile(q).astype(int) for c in ["raw", "clean", "noprior"]})
tab.index = [f"p{int(x*100)}" for x in q]
print(tab.T.to_string())
for c in ["raw", "clean", "noprior"]:
    print(f"truncated at {MAX_TOK} tokens ({c:<7}): {(df['ntok_' + c] > MAX_TOK).mean()*100:5.1f}%")
df["over_cap"] = df["ntok_clean"] > MAX_TOK

fig, ax = plt.subplots(figsize=(7, 3))
bins = np.arange(0, 330, 10)
ax.hist(df["ntok_raw"].clip(upper=320), bins=bins, color="#86b6ef", label="raw")
ax.hist(df["ntok_clean"].clip(upper=320), bins=bins, color="#1c5cab", alpha=.85, label="clean")
ax.axvline(MAX_TOK, color="#eb6834", lw=2, label=f"cap = {MAX_TOK}")
ax.set_xlabel(f"impression length ({TOKENIZER} tokens)"); ax.set_ylabel("reports"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## Step 5 — Images → 512 × 512 `uint8`

Each PNG is read at full size, scaled so its longer side is 512 px with **area interpolation** (averages the
source pixels, so no aliasing from ribs or grid lines), and centred on a zero background — the aspect ratio
(~1.22 : 1) is kept, so the lung apices and costophrenic angles are not cropped. All images go into one
memory-mapped array, which a dataloader can read without any PNG decoding. The scale and padding are stored per
image, so coordinates can be mapped back to the original.

In [ ]:
def resize_pad(path):
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im is None:
        raise IOError(path)
    if im.ndim == 3:
        im = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    if im.dtype != np.uint8:                         # not expected (EDA: all 8-bit L) — scale defensively
        im = cv2.normalize(im, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    h, w = im.shape
    s = SIZE / max(h, w)
    nh, nw = max(1, round(h * s)), max(1, round(w * s))
    small = cv2.resize(im, (nw, nh), interpolation=cv2.INTER_AREA)
    out = np.zeros((SIZE, SIZE), np.uint8)
    y0, x0 = (SIZE - nh) // 2, (SIZE - nw) // 2
    out[y0:y0 + nh, x0:x0 + nw] = small
    return out, (h, w, s, x0, y0, nw, nh, float(im.mean()))

os.makedirs(WORK, exist_ok=True)
NPY = os.path.join(WORK, "images_512.npy")
arr = np.lib.format.open_memmap(NPY, mode="w+", dtype=np.uint8, shape=(len(df), SIZE, SIZE))
geo = [None] * len(df)
t0 = time.time()
with Pool(os.cpu_count()) as pool:
    for i, (img, g) in enumerate(pool.imap(resize_pad, df["abs_path"], chunksize=8)):
        arr[i] = img
        geo[i] = g
        if (i + 1) % 1000 == 0:
            print(f"{i+1:>6,}/{len(df):,}  {time.time()-t0:6.0f}s", flush=True)
arr.flush()
geo = pd.DataFrame(geo, columns=["orig_h", "orig_w", "scale", "pad_x", "pad_y", "new_w", "new_h", "orig_mean"])
df = pd.concat([df, geo], axis=1)
print(f"Done in {time.time()-t0:.0f}s → {NPY}  ({os.path.getsize(NPY)/1e9:.2f} GB)")

**Verify the array**: shape, no blank images, intensity preserved by the resize, and a visual comparison.

In [ ]:
a = np.load(NPY, mmap_mode="r")
assert a.shape == (len(df), SIZE, SIZE) and a.dtype == np.uint8
# mean over the image region only (padding excluded) should match the original's mean
inner = np.array([a[i, r.pad_y:r.pad_y + r.new_h, r.pad_x:r.pad_x + r.new_w].mean() for i, r in df.iterrows()])
dmean = np.abs(inner - df["orig_mean"].to_numpy())
blank = int((a.reshape(len(df), -1).max(axis=1) == 0).sum())
print(f"array {a.shape} {a.dtype} | blank images: {blank} | |mean(512) − mean(original)|: "
      f"median {np.median(dmean):.3f}, max {dmean.max():.3f} (grey levels of 255)")
assert blank == 0 and dmean.max() < 2.0
print("Original sizes (h × w):", df.groupby(["orig_h", "orig_w"]).size().sort_values(ascending=False).head(5).to_dict())

samp = df.groupby("label").sample(3, random_state=SEED)
fig, ax = plt.subplots(2, 6, figsize=(15, 5.6))
for j, (_, r) in enumerate(samp.iterrows()):
    ax[0, j].imshow(cv2.imread(r.abs_path, cv2.IMREAD_GRAYSCALE), cmap="gray")
    ax[0, j].set_title(f"original {r.orig_w}×{r.orig_h}", fontsize=8)
    ax[1, j].imshow(a[r.idx], cmap="gray", vmin=0, vmax=255)
    ax[1, j].set_title(f"512 stored ({r.label})", fontsize=8)
for x in ax.flat:
    x.axis("off")
plt.tight_layout(); plt.show()

## Step 6 — Label targets and masks

From `chexbert_impression.csv` (labels of the same text the model learns to generate):
- `y_<label>` = 1 if positive, else 0 (negative **and** not mentioned → 0, the standard CheXpert convention)
- `m_<label>` = 0 where the label is uncertain (−1) so the clinical-consistency loss ignores it (U-Ignore), else 1
- the raw values (1 / 0 / −1 / blank) are kept under the original names

Pneumonia and Pleural Other are kept as targets but excluded from per-class test reporting (≈14 and 10 test
positives); they are listed in `cleaning_summary.json`.

In [ ]:
def col(l):
    return re.sub(r"\W+", "_", l.lower()).strip("_")

for l in LABELS:
    df["y_" + col(l)] = (df[l] == 1).astype(np.int8)
    df["m_" + col(l)] = (df[l] != -1).astype(np.int8)

summ = pd.DataFrame({
    "positive": [int((df[l] == 1).sum()) for l in LABELS],
    "uncertain (masked)": [int((df[l] == -1).sum()) for l in LABELS],
    "test positives": [int(((df[l] == 1) & (df.subset == "test")).sum()) for l in LABELS],
}, index=LABELS)
print(summ.to_string())

# Spot-check from the EDA: images labelled No Finding that also carry a positive label — which labels?
nf = df["No Finding"] == 1
co = {l: int(((df[l] == 1) & nf).sum()) for l in PATHS if ((df[l] == 1) & nf).any()}
print(f"\nNo Finding = 1 and some positive label: {int((nf & (df[PATHS] == 1).any(axis=1)).sum())} images → {co}")
print("(CheXpert's No Finding ignores Support Devices, so devices are expected here; anything else is a label conflict.)")

## Save

In [ ]:
RARE = ["Pneumonia", "Pleural Other"]
cols = (["idx", "local_path", "path_to_image", "deid_patient_id", "subset", "label", "age", "sex",
         "impression_raw", "impression_clean", "impression_noprior",
         "has_prior", "noprior_residual", "noprior_empty", "ntok_raw", "ntok_clean", "ntok_noprior", "over_cap"]
        + LABELS + [p + col(l) for l in LABELS for p in ("y_", "m_")]
        + ["orig_h", "orig_w", "scale", "pad_x", "pad_y", "new_w", "new_h"])
cols = [c for c in cols if c in df.columns]
df[cols].to_csv(os.path.join(WORK, "reports_clean.csv"), index=False)
dropped.to_csv(os.path.join(WORK, "dropped_rows.csv"), index=False)

summary = {
    "source": "IE643 Project v3 (jithendraampatt/ie643-project)",
    "rows_in": 10000, "rows_kept": int(len(df)), "rows_dropped": int(len(dropped)),
    "split_counts": df["subset"].value_counts().reindex(["train", "val", "test"]).astype(int).to_dict(),
    "image_array": {"file": "images_512.npy", "shape": [int(x) for x in a.shape], "dtype": "uint8",
                    "resize": "longer side to 512, cv2.INTER_AREA, centred zero padding"},
    "text": {"target": "impression_clean", "ablation": "impression_noprior", "tokenizer": TOKENIZER,
             "max_tokens": MAX_TOK, "boilerplate_hits": dict(hits),
             "pct_has_prior": round(float(df.has_prior.mean() * 100), 2),
             "pct_over_cap_clean": round(float(df.over_cap.mean() * 100), 2)},
    "labels": {"source": "chexbert_impression.csv", "columns": LABELS,
               "target_rule": "y = 1 if label == 1 else 0", "mask_rule": "m = 0 if label == -1 (uncertain) else 1",
               "exclude_from_per_class_reporting": RARE},
}
json.dump(summary, open(os.path.join(WORK, "cleaning_summary.json"), "w"), indent=2)
for f in sorted(os.listdir(WORK)):
    p = os.path.join(WORK, f)
    if os.path.isfile(p):
        print(f"{f:<24} {os.path.getsize(p)/1e6:10.1f} MB")
print(json.dumps(summary, indent=2))

## Loading this in later notebooks

```python
import numpy as np, pandas as pd
D = "/kaggle/input/<this notebook's output folder>"
meta = pd.read_csv(f"{D}/reports_clean.csv")
imgs = np.load(f"{D}/images_512.npy", mmap_mode="r")     # imgs[meta.idx[i]] is meta row i
train = meta[meta.subset == "train"]
```